# Linear Acceleration — SUVAT Visualiser

Motion in a straight line with **constant acceleration** is described by the SUVAT equations. For an initial velocity $u$, a constant acceleration $a$ and a time $t$:

$$v = u + at$$

$$s = ut + \tfrac{1}{2}at^2$$

Use the sliders below to set $u$, $a$ and $t$, then press **Run** to update the graphs and the animation.

- The **displacement–time** graph shows $s(t) = ut + \tfrac{1}{2}at^2$ — a parabola when $a \ne 0$, a straight line when $a = 0$.
- The **velocity–time** graph shows $v(t) = u + at$ — always a straight line. Its **slope is the acceleration** $a$, and the **area under the line is the displacement** $s$ (shaded).

The animation has its own **play / pause** button and a **time slider** (generated by `FuncAnimation`). Press play and watch the marker trace out both graphs, or drag the time slider to inspect a particular instant.


In [ ]:
# Install ipywidgets into the in-browser (Pyodide) kernel.
# numpy, matplotlib and pandas are bundled with Pyodide, but ipywidgets is not,
# so it is fetched at runtime. Re-running this cell is harmless (cached).
import micropip
await micropip.install("ipywidgets")


In [1]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation

import ipywidgets as widgets

from IPython.display import HTML, display


def make_suvat_animation(u, a, t_total, n_frames=80):
    """Build the two-panel SUVAT visualiser and return it as an in-browser animation."""

    t = np.linspace(0, t_total, n_frames)
    s = u * t + 0.5 * a * t ** 2
    v = u + a * t

    fig, (ax_s, ax_v) = plt.subplots(1, 2, figsize=(11, 4.2))

    # ---- displacement-time graph ----
    ax_s.set_xlim(0, t_total)
    ax_s.set_ylim(min(0.0, s.min()) * 1.15, max(0.0, s.max()) * 1.15)
    ax_s.set_xlabel("Time, t (s)")
    ax_s.set_ylabel("Displacement, s (m)")
    ax_s.set_title("Displacement–time")
    ax_s.axhline(0, color="grey", linewidth=0.8)
    ax_s.grid(alpha=0.3)

    # ---- velocity-time graph ----
    ax_v.set_xlim(0, t_total)
    ax_v.set_ylim(min(0.0, v.min()) * 1.15, max(0.0, v.max()) * 1.15)
    ax_v.set_xlabel("Time, t (s)")
    ax_v.set_ylabel("Velocity, v (m/s)")
    ax_v.set_title("Velocity–time  (shaded area = displacement)")
    ax_v.axhline(0, color="grey", linewidth=0.8)
    ax_v.grid(alpha=0.3)

    # Faint full curves showing what the motion will trace out.
    ax_s.plot(t, s, color="tab:blue", alpha=0.25, linewidth=1.5)
    ax_v.plot(t, v, color="tab:red", alpha=0.25, linewidth=1.5)

    # Progressive bold lines and moving markers.
    line_s, = ax_s.plot([], [], color="tab:blue", linewidth=2.5)
    line_v, = ax_v.plot([], [], color="tab:red", linewidth=2.5)
    dot_s, = ax_s.plot([], [], "o", color="tab:blue", ms=8)
    dot_v, = ax_v.plot([], [], "o", color="tab:red", ms=8)

    # Progressive shaded area under the velocity line (= displacement so far).
    fill_v = ax_v.fill_between([], [], [], alpha=0.25, color="tab:red")

    # Readout showing the current time, displacement and velocity.
    readout = fig.text(
        0.5, 0.02, "",
        ha="center", va="bottom", fontsize=11,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.9),
    )

    def update(i):
        tt = t[:i + 1]
        line_s.set_data(tt, s[:i + 1])
        line_v.set_data(tt, v[:i + 1])
        dot_s.set_data([t[i]], [s[i]])
        dot_v.set_data([t[i]], [v[i]])
        fill_v.set_data(tt, np.zeros_like(tt), v[:i + 1])
        readout.set_text(
            f"t = {t[i]:.2f} s    s = {s[i]:.2f} m    v = {v[i]:.2f} m/s"
        )
        return line_s, line_v, dot_s, dot_v, fill_v, readout

    fig.tight_layout(rect=[0, 0.08, 1, 1])
    anim = FuncAnimation(fig, update, frames=n_frames, interval=50, repeat=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


In [3]:
# ============================================================
# Interactive controls: set u, a and t, then press Run.
# ============================================================

u_slider = widgets.FloatSlider(
    value=5, min=0, max=30, step=0.5,
    description="u (m/s):",
    continuous_update=False,
)

a_slider = widgets.FloatSlider(
    value=2, min=-10, max=10, step=0.5,
    description="a (m/s²):",
    continuous_update=False,
)

t_slider = widgets.FloatSlider(
    value=6, min=0.5, max=20, step=0.5,
    description="t (s):",
    continuous_update=False,
)

run_button = widgets.Button(description="Run", button_style="primary", icon="play")
value_label = widgets.HTML()
anim_output = widgets.Output(layout=widgets.Layout(width="900px"))


def show_values():
    u = u_slider.value
    a = a_slider.value
    t = t_slider.value
    v = u + a * t
    s = u * t + 0.5 * a * t ** 2
    value_label.value = (
        f"<b>u</b> = {u:g} m/s &nbsp;&nbsp; <b>a</b> = {a:g} m/s² &nbsp;&nbsp; "
        f"<b>t</b> = {t:g} s<br><br>"
        f"<b>Final velocity</b> &nbsp; v = u + at = {v:.2f} m/s<br>"
        f"<b>Displacement</b> &nbsp;&nbsp;&nbsp; s = ut + ½at² = {s:.2f} m"
    )


def run_clicked(button):
    show_values()
    with anim_output:
        anim_output.clear_output(wait=True)
        display(make_suvat_animation(u_slider.value, a_slider.value, t_slider.value))


run_button.on_click(run_clicked)

controls = widgets.VBox(
    [
        widgets.HTML("<h3>Parameters</h3>"),
        u_slider,
        a_slider,
        t_slider,
        run_button,
        value_label,
    ],
    layout=widgets.Layout(width="340px"),
)

display(widgets.VBox(
    [controls, anim_output],
    layout=widgets.Layout(align_items="flex-start", gap="18px"),
))

show_values()
# with anim_output:
#     display(make_suvat_animation(u_slider.value, a_slider.value, t_slider.value))
